# 03 · Proyecto final: selección de modelos con tickets reales

**Objetivo empresarial.** Comparar Gemini y un modelo multimodal local usando el mismo contrato,
ground truth y métricas: exactitud, latencia, tokens y costo estimado.

**Ruta en la presentación:** Bloque B (selección/costos) → Bloque D (evaluación y decisión).

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from receipt_validation.config import load_settings

settings = load_settings(ROOT)
print(f"Project root: {ROOT}")

Project root: /Users/andrei/Documents/Projects/Curso LLM Codes/session_2_architecture


In [2]:
import json
import pandas as pd
from IPython.display import display

from receipt_validation.clients import ask_model, ask_model_raw, parse_json_content
from receipt_validation.cost import estimate_cost
from receipt_validation.evaluators import evaluate_receipt, summarize_results
from receipt_validation.experiments import run_comparison, save_results
from receipt_validation.io import load_expected_receipts, load_receipt_images
from receipt_validation.prompts import GENERIC_PROMPT, QUESTIONS_PROMPT, build_extraction_prompt
from receipt_validation.schemas import ReceiptExtraction

paths = settings["paths"]
RUN_PROVIDER_CALLS = True
DATASET = "test"  # Use "eval" only for the final unbiased comparison.
labels_file = paths.test_labels_file if DATASET == "test" else paths.eval_labels_file
images_dir = paths.test_receipts_dir if DATASET == "test" else paths.eval_receipts_dir

## 1. Validar datos antes de gastar tokens

La prueba se detiene si el CSV está mal formado o falta una imagen. El conjunto `test` permite
ajustar prompts; `eval` se reserva para la comparación final y evita optimizar contra el examen.

In [3]:
expected = load_expected_receipts(labels_file)
images = load_receipt_images(images_dir)
image_names = {path.name for path in images}
missing_images = sorted(set(expected["file_name"]) - image_names)
assert not missing_images, f"Missing images: {missing_images}"
display(expected.head())
print(f"Rows: {len(expected)} | Images: {len(images)} | Dataset: {DATASET}")

,file_name,fecha,folio,rfc_emisor,estacion,moneda,monto_total,valid_total,producto,cantidad_litros,precio_unitario,notes
0,ticket_1.jpg,2025-06-03 22:06,1174049,SCC111213TG9,Servicio Gasolinero de la Costa S.A. de C.V.,MXN,1571.82,True,Magna,65.005000,24.18,RFC/header partially unreadable.
1,ticket_2.jpg,2025-06-09 17:46,530036,NaN,NaN,MXN,370.63,True,Premium,14.636000,24.90,Folio/date/header are low confidence due dista...
2,ticket_3.jpg,2025-06-04 11:18,GT8691,GTE941124DN6,Gasolinera Teoloyucan S.A. de C.V.,MXN,5000.00,True,Diesel UltrabaJo,195.388000,25.59,Quantity and unit price are difficult to read;...
3,ticket_4.jpg,2025-06-04 11:52,GT6693,GTE941124DN6,Gasolinera Teoloyucan S.A. de C.V.,MXN,7000.00,True,Diesel UltrabaJo,273.544353,25.59,Payment voucher also visible.
4,ticket_5.jpg,2025-06-03 17:35:24,361841,ESD04101425A,Servicio del Desierto,MXN,1760.00,True,Diesel 34006,64.250000,27.39,RFC appears as ESD041014...? verify exact RFC.


Rows: 10 | Images: 10 | Dataset: test


## 2. Experimentos de prompt

Comparamos cuatro variantes: pregunta genérica, extracción zero-shot estructurada, few-shot y
preguntas guiadas. Solo cambia el prompt; imagen, modelo y evaluación permanecen constantes.

In [4]:
from IPython.display import Markdown

prompt_variants = {
    "generic": GENERIC_PROMPT,
    "structured_zero_shot": build_extraction_prompt(use_few_shot=False),
    "structured_few_shot": build_extraction_prompt(use_few_shot=True),
    "structured_questions": QUESTIONS_PROMPT,
}
for name, prompt in prompt_variants.items():
    display(Markdown(
        f"#### `{name}` · {len(prompt)} caracteres\n\n```text\n{prompt}\n```\n\n---"
    ))

#### `generic` · 146 caracteres

```text
Give me from this ticket the following fields: fecha, folio, rfc_emisor, estacion,
moneda, monto_total, productos and validation. Respond in JSON.
```

---

#### `structured_zero_shot` · 763 caracteres

```text
Analyze this gasoline or purchase receipt.

Extract these fields:
- fecha: ISO date if possible, otherwise the visible date as text.
- folio: receipt or invoice identifier.
- rfc_emisor: Mexican RFC for the issuer.
- estacion: station or merchant name.
- moneda: currency code, usually MXN.
- monto_total: final total paid.
- productos: each product with description, liters/quantity when available, unit price, and amount.
- validation: consistency checks for total, RFC format, date format, and any issues.

Business rules:
- If a value is missing or unreadable, use null.
- Do not approximate fiscal fields.
- If product amounts do not sum to the total, report the issue in validation.issues.
- Return only valid JSON. No Markdown. No explanation outside JSON.
```

---

#### `structured_few_shot` · 1220 caracteres

```text
Analyze this gasoline or purchase receipt.

Extract these fields:
- fecha: ISO date if possible, otherwise the visible date as text.
- folio: receipt or invoice identifier.
- rfc_emisor: Mexican RFC for the issuer.
- estacion: station or merchant name.
- moneda: currency code, usually MXN.
- monto_total: final total paid.
- productos: each product with description, liters/quantity when available, unit price, and amount.
- validation: consistency checks for total, RFC format, date format, and any issues.

Business rules:
- If a value is missing or unreadable, use null.
- Do not approximate fiscal fields.
- If product amounts do not sum to the total, report the issue in validation.issues.
- Return only valid JSON. No Markdown. No explanation outside JSON.

Example output:
{
  "fecha": "2026-06-03",
  "folio": "A-88213",
  "rfc_emisor": "PEM980101ABC",
  "estacion": "Pemex 3841",
  "moneda": "MXN",
  "monto_total": 982.50,
  "productos": [
    {
      "descripcion": "Magna",
      "cantidad_litros": 42.3,
      "precio_unitario": 23.22,
      "monto": 982.50
    }
  ],
  "validation": {
    "total_matches_products": true,
    "rfc_format_valid": true,
    "date_format_valid": true,
    "issues": []
  }
}
```

---

#### `structured_questions` · 1090 caracteres

```text
Look at this receipt image and answer the following questions:

1. What is the purchase date? (fecha — ISO format if possible, otherwise the visible date)
2. What is the receipt or invoice number? (folio)
3. What is the issuer's Mexican RFC? (rfc_emisor)
4. What is the name of the station or merchant? (estacion)
5. What currency was used? (moneda — usually MXN)
6. What was the final total paid? (monto_total — a number, not a string)
7. Which products appear on the receipt? (productos — for each one: descripcion,
   cantidad_litros when available, precio_unitario, monto)
8. Do the product amounts add up to the total? Is the RFC format valid? Is the date
   format valid? Report any inconsistency. (validation — total_matches_products,
   rfc_format_valid, date_format_valid, issues)

Rules: if an answer is missing or unreadable, use null. Never guess fiscal fields.

Answer ALL questions in one single JSON object with exactly these keys:
fecha, folio, rfc_emisor, estacion, moneda, monto_total, productos, validation.
Return only the JSON — no explanations, no Markdown, just JSON.
```

---

### Demo: evaluar los prompts con 5 tickets (modo crudo, sin esquema forzado)

Antes de la matriz completa, evaluamos las cuatro variantes de prompt con 5 tickets para **ver**
cuánto importa el prompt cuando nada más lo rescata.

**Clave:** esta demo usa `ask_model_raw` — **sin** salida estructurada (`responseJsonSchema` /
`response_format`) y **sin** el system prompt de extracción. El prompt de usuario es lo único que
guía al modelo. Con el prompt genérico el modelo tiende a inventar formatos o devolver JSON
malformado; con los estructurados, las reglas y el cierre "return only JSON" garantizan salida
parseable. (La matriz completa y el dashboard sí usan esquema forzado, como en producción.)

Escalera de prompts:

| Variante | Qué agrega |
|---|---|
| `generic` | Solo campos + "respond in JSON" |
| `structured_zero_shot` | + reglas de formato, null, reglas de negocio |
| `structured_few_shot` | + ejemplo de salida |
| `structured_questions` | Preguntas numeradas por campo → cierre exigiendo un solo JSON |

- `DEMO_USE_CLOUD = False` → modelo local (LM Studio). `True` → Gemini en la nube.
- En la nube, el cliente limita automáticamente las peticiones por minuto:
  **15/min** para `gemini-3.1-flash-lite` y **5/min** para cualquier otro modelo
  (5 tickets × 4 prompts = 20 llamadas, ~1.5 min con flash-lite; más con otro modelo).

In [ ]:
RUN_PROMPT_DEMO = True  # Activar solo con llaves/servidor listos.
DEMO_USE_CLOUD = True   # False = LM Studio local | True = Gemini en la nube
DEMO_TICKETS = 5

DEMO_BACKEND = "gemini" if DEMO_USE_CLOUD else "lmstudio"
DEMO_MODEL = (
    settings["default_gemini_model"] if DEMO_USE_CLOUD else settings["default_lmstudio_model"]
)

if DEMO_USE_CLOUD:
    from receipt_validation.clients import gemini_rate_limit
    print(f"Rate limit para {DEMO_MODEL}: {gemini_rate_limit(DEMO_MODEL)} peticiones/minuto")

demo_results = []
if RUN_PROMPT_DEMO:
    demo_rows = expected.head(DEMO_TICKETS)
    total_calls = len(demo_rows) * len(prompt_variants)
    call_number = 0
    for _, row in demo_rows.iterrows():
        image_path = images_dir / row["file_name"]
        for variant_name, prompt in prompt_variants.items():
            call_number += 1
            print(f"[{call_number:02d}/{total_calls:02d}] {row['file_name']} · {variant_name}")
            record = {
                "prompt": variant_name,
                "file_name": row["file_name"],
                "accuracy": 0.0,
                "passed": False,
                "latency_seconds": None,
                "error": None,
                "extraction": None,
            }
            try:
                response = ask_model_raw(
                    backend=DEMO_BACKEND,
                    prompt=prompt,
                    image_path=image_path,
                    model=DEMO_MODEL,
                    settings=settings,
                )
                record["latency_seconds"] = round(response.latency_seconds, 2)
                extraction = ReceiptExtraction.model_validate(
                    parse_json_content(response.content)
                )
                evaluation = evaluate_receipt(extraction, row)
                record["accuracy"] = evaluation["accuracy"]
                record["passed"] = evaluation["passed"]
                record["extraction"] = extraction.model_dump()
            except Exception as exc:
                record["error"] = f"{type(exc).__name__}: {exc}"[:150]
            demo_results.append(record)
else:
    print("Demo disabled. Set RUN_PROMPT_DEMO=True when ready.")

if demo_results:
    demo_frame = pd.DataFrame(demo_results).drop(columns=["extraction"], errors="ignore")
    pivot = demo_frame.pivot_table(index="file_name", columns="prompt", values="accuracy")
    display(pivot.style.format("{:.1%}").background_gradient(cmap="YlGn", axis=None))
    display(
        demo_frame.groupby("prompt", as_index=False)
        .agg(
            accuracy=("accuracy", "mean"),
            tickets_ok=("passed", "sum"),
            avg_latency_seconds=("latency_seconds", "mean"),
            errores=("error", lambda column: column.notna().sum()),
        )
        .sort_values("accuracy", ascending=False)
    )

Rate limit para gemini-3.1-flash-lite: 15 peticiones/minuto
[01/20] ticket_1.jpg · generic
[02/20] ticket_1.jpg · structured_zero_shot
[03/20] ticket_1.jpg · structured_few_shot
[04/20] ticket_1.jpg · structured_questions
[05/20] ticket_2.jpg · generic
[06/20] ticket_2.jpg · structured_zero_shot
[07/20] ticket_2.jpg · structured_few_shot
[08/20] ticket_2.jpg · structured_questions
[09/20] ticket_3.jpg · generic
[10/20] ticket_3.jpg · structured_zero_shot
[11/20] ticket_3.jpg · structured_few_shot
[12/20] ticket_3.jpg · structured_questions
[13/20] ticket_4.jpg · generic
[14/20] ticket_4.jpg · structured_zero_shot
[15/20] ticket_4.jpg · structured_few_shot
[16/20] ticket_4.jpg · structured_questions
[17/20] ticket_5.jpg · generic
[18/20] ticket_5.jpg · structured_zero_shot
[19/20] ticket_5.jpg · structured_few_shot
[20/20] ticket_5.jpg · structured_questions


prompt,generic,structured_few_shot,structured_questions,structured_zero_shot
file_name,,,,
ticket_1.jpg,50.0%,66.7%,66.7%,66.7%
ticket_2.jpg,66.7%,66.7%,66.7%,66.7%
ticket_3.jpg,66.7%,83.3%,83.3%,83.3%
ticket_4.jpg,33.3%,50.0%,50.0%,50.0%
ticket_5.jpg,83.3%,83.3%,83.3%,100.0%


,prompt,accuracy,tickets_ok,avg_latency_seconds,errores
3,structured_zero_shot,0.733333,1,1.866,0
1,structured_few_shot,0.700000,0,1.592,0
2,structured_questions,0.700000,0,8.716,0
0,generic,0.600000,0,1.602,0


### ¿Extrae bien o es ilusión? Valor esperado vs. extraído, campo por campo

La tabla de accuracy resume; esta tabla muestra la evidencia. Por cada ticket: la fila
**esperado** (ground truth del CSV) y una fila por variante de prompt con los valores que
ese JSON trajo realmente. Verde = coincide con lo esperado, rojo = no coincide.
La `fecha` se compara **ignorando la hora** (el CSV la trae con hora y el modelo suele
devolver solo la fecha) — así el semáforo refleja al prompt, no al formato de las etiquetas.


In [6]:
import re

FIELDS = ["fecha", "folio", "rfc_emisor", "estacion", "moneda", "monto_total"]
DATE_PATTERN = re.compile(r"\d{4}-\d{2}-\d{2}")

def _norm(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return str(value).strip().lower()

if demo_results and all(record.get("extraction") is None for record in demo_results):
    print(
        "demo_results no trae los JSON extraidos (vienen de una corrida anterior).\n"
        "Re-ejecuta la celda de la demo y luego esta celda."
    )
elif demo_results:
    comparison_rows = []
    for _, row in demo_rows.iterrows():
        comparison_rows.append({
            "file_name": row["file_name"],
            "fuente": "esperado",
            **{field: row.get(field) for field in FIELDS},
        })
        for record in demo_results:
            if record["file_name"] != row["file_name"]:
                continue
            extraction = record.get("extraction") or {}
            comparison_rows.append({
                "file_name": record["file_name"],
                "fuente": record["prompt"],
                **{field: extraction.get(field) for field in FIELDS},
            })
    comparison = pd.DataFrame(comparison_rows)
    expected_by_file = {
        item["file_name"]: item for item in comparison_rows if item["fuente"] == "esperado"
    }

    def _matches(field, expected_value, extracted_value):
        if field == "fecha":
            expected_date = DATE_PATTERN.search(_norm(expected_value))
            extracted_date = DATE_PATTERN.search(_norm(extracted_value))
            if expected_date and extracted_date:
                return expected_date.group() == extracted_date.group()
        if field == "monto_total":
            try:
                return abs(float(expected_value) - float(extracted_value)) <= 0.05
            except (TypeError, ValueError):
                pass
        expected_text = _norm(expected_value)
        return expected_text == "" or expected_text == _norm(extracted_value)

    def _highlight(frame):
        styles = pd.DataFrame("", index=frame.index, columns=frame.columns)
        for i, r in frame.iterrows():
            if r["fuente"] == "esperado":
                styles.loc[i, :] = "background-color:#fff3cd; color:#000; font-weight:bold"
                continue
            expected_row = expected_by_file[r["file_name"]]
            for field in FIELDS:
                ok = _matches(field, expected_row[field], r[field])
                styles.loc[i, field] = (
                    "background-color:#d4edda; color:#000" if ok
                    else "background-color:#f8d7da; color:#000"
                )
        return styles

    display(comparison.style.apply(_highlight, axis=None).hide(axis="index"))
else:
    print("Corre primero la demo (RUN_PROMPT_DEMO=True) para llenar demo_results.")

file_name,fuente,fecha,folio,rfc_emisor,estacion,moneda,monto_total
ticket_1.jpg,esperado,2025-06-03 22:06,1174049,SCC111213TG9,Servicio Gasolinero de la Costa S.A. de C.V.,MXN,1571.820000
ticket_1.jpg,generic,03/06/2025,1174049,SGC111213TG9,PL/20676/EXP/ES/2017,MXN,1571.820000
ticket_1.jpg,structured_zero_shot,2025-06-03,1174049,SGC111213TG9,"SERVICIO GASOLINERO DE LA COSTA, S.A. DE",MXN,1571.820000
ticket_1.jpg,structured_few_shot,2025-06-03,1174049,SGC111213TG9,"SERVICIO GASOLINERO DE LA COSTA, S.A. DE",MXN,1571.820000
ticket_1.jpg,structured_questions,2025-06-03,1174049,SGC111213TG9,"SERVICIO GASOLINERO DE LA COSTA, S.A. DE",MXN,1571.820000
ticket_2.jpg,esperado,2025-06-09 17:46,530036,nan,nan,MXN,370.630000
ticket_2.jpg,generic,04/06/2025,530036,GAF220603TC4,E14919,MXN,370.830000
ticket_2.jpg,structured_zero_shot,2025-06-04,530036,GAF220603TC4,GAFSACOMM,MXN,370.830000
ticket_2.jpg,structured_few_shot,2025-06-04,530036,GAF220603TC4,GAFSACOMM,MXN,370.830000
ticket_2.jpg,structured_questions,2025-06-04,530036,GAF220603TC4,GAFSACOMM,MXN,370.830000


### Lectura de resultados: modelos de nube y prompts sencillos

Para modelos en la nube como `gemini-3.1-flash-lite`, en tareas sencillas de extracción el
desempeño ya es bueno **incluso con un prompt sencillo**: en la tabla se ve que el `generic`
extrae folio, RFC, moneda y monto prácticamente igual que las variantes estructuradas.

Donde falla es en lo específico. El ejemplo claro es la **fecha**: todas las variantes logran
extraerla, pero el `generic` la regresa como `03/06/2025` mientras las estructuradas regresan
`2025-06-03` — por no decirle **cómo la esperamos**, el modelo elige el formato que quiere.
Lo mismo pasa con el nombre de la estación (mayúsculas, abreviaturas, texto extra) y con
RFCs confundidos por caracteres parecidos en la imagen.

Esa es la lección general: cuando no se especifica bien lo que queremos, el modelo no falla
en "sí extrae o no extrae", falla en los detalles del formato — y esos detalles son los que
rompen sistemas: una fecha en otro formato no parsea, un RFC con un carácter cambiado no
valida, un nombre con texto extra no hace match. El prompt estructurado no mejora tanto la
*extracción*; lo que garantiza es la *consistencia* del resultado.


## 3. Un contrato para todos los modelos

Validar JSON separa “el modelo respondió” de “el sistema recibió datos utilizables”.
Los errores se guardan como resultados; no se pierden silenciosamente.

In [7]:
SAMPLE_JSON = {
    "fecha": "2025-06-04",
    "folio": "GT6693",
    "rfc_emisor": "GTE941124DN6",
    "estacion": "Gasolinera Teoloyucan S.A. de C.V.",
    "moneda": "MXN",
    "monto_total": 7000.0,
    "productos": [],
    "validation": {"total_matches_products": None, "rfc_format_valid": True,
                   "date_format_valid": True, "issues": []},
}
ReceiptExtraction.model_validate(SAMPLE_JSON).model_dump()

{'fecha': '2025-06-04',
 'folio': 'GT6693',
 'rfc_emisor': 'GTE941124DN6',
 'estacion': 'Gasolinera Teoloyucan S.A. de C.V.',
 'moneda': 'MXN',
 'monto_total': 7000.0,
 'productos': [],
 'validation': {'total_matches_products': None,
  'rfc_format_valid': True,
  'date_format_valid': True,
  'issues': []}}

## 4. Ejecutar la matriz experimental

> **Pausa:** estimen primero qué combinación ganará en calidad, costo y latencia.
Mantengan `RUN_PROVIDER_CALLS=False` durante la explicación; actívenlo solo con llaves/servidor listos.

In [8]:
def run_full_matrix(max_receipts: int | None = None) -> pd.DataFrame:
    models = [
        {"backend": "gemini", "model": settings["default_gemini_model"]},
        {"backend": "lmstudio", "model": settings["default_lmstudio_model"]},
    ]
    pipelines = ["llm_only", "ocr_llm"]

    def show_progress(current: int, total: int, message: str) -> None:
        print(f"[{current:02d}/{total:02d}] {message}")

    return run_comparison(
        expected=expected,
        images_dir=images_dir,
        models=models,
        pipelines=pipelines,
        settings=settings,
        use_few_shot=False,
        max_receipts=max_receipts,
        dataset_name=DATASET,
        progress_callback=show_progress,
    )

In [9]:
results_frame = pd.DataFrame()
if RUN_PROVIDER_CALLS:
    results_frame = run_full_matrix()
else:
    print("Provider calls are disabled. Set RUN_PROVIDER_CALLS=True when ready.")

[01/40] ticket_1.jpg · llm_only · gemini/gemini-3.1-flash-lite
[02/40] ticket_1.jpg · llm_only · lmstudio/google/gemma-4-e2b
[03/40] ticket_1.jpg · ocr_llm · gemini/gemini-3.1-flash-lite
[04/40] ticket_1.jpg · ocr_llm · lmstudio/google/gemma-4-e2b
[05/40] ticket_2.jpg · llm_only · gemini/gemini-3.1-flash-lite
[06/40] ticket_2.jpg · llm_only · lmstudio/google/gemma-4-e2b
[07/40] ticket_2.jpg · ocr_llm · gemini/gemini-3.1-flash-lite
[08/40] ticket_2.jpg · ocr_llm · lmstudio/google/gemma-4-e2b
[09/40] ticket_3.jpg · llm_only · gemini/gemini-3.1-flash-lite
[10/40] ticket_3.jpg · llm_only · lmstudio/google/gemma-4-e2b
[11/40] ticket_3.jpg · ocr_llm · gemini/gemini-3.1-flash-lite
[12/40] ticket_3.jpg · ocr_llm · lmstudio/google/gemma-4-e2b
[13/40] ticket_4.jpg · llm_only · gemini/gemini-3.1-flash-lite
[14/40] ticket_4.jpg · llm_only · lmstudio/google/gemma-4-e2b
[15/40] ticket_4.jpg · ocr_llm · gemini/gemini-3.1-flash-lite
[16/40] ticket_4.jpg · ocr_llm · lmstudio/google/gemma-4-e2b
[17/40] 

## 5. Tabla, visualización y exportación

La tabla responde qué modelo conviene. El CSV detallado alimenta el dashboard sin volver a gastar
tokens y conserva evidencia por ticket para investigar fallos.

In [10]:
if not results_frame.empty:
    summary = (
        results_frame.groupby(["pipeline", "backend", "model"], as_index=False)
        .agg(
            accuracy=("accuracy", "mean"),
            avg_latency_seconds=("latency_seconds", "mean"),
            avg_input_tokens=("input_tokens", "mean"),
            avg_output_tokens=("output_tokens", "mean"),
            total_cost=("estimated_cost", "sum"),
        )
    )
    display(summary.style.format({
        "accuracy": "{:.1%}",
        "avg_latency_seconds": "{:.2f}",
        "total_cost": "${:.6f}",
    }).background_gradient(subset=["accuracy"], cmap="YlGn"))
    display(results_frame.pivot_table(
        index="file_name", columns=["pipeline", "backend"], values="accuracy"
    ))
    output_path = save_results(results_frame, paths.outputs_dir)
    print(f"Saved: {output_path}")
else:
    summary = pd.DataFrame()
    display(summary)

,pipeline,backend,model,accuracy,avg_latency_seconds,avg_input_tokens,avg_output_tokens,total_cost
0,llm_only,gemini,gemini-3.1-flash-lite,73.3%,2.15,1297.800000,213.200000,$0.006443
1,llm_only,lmstudio,google/gemma-4-e2b,51.7%,3.66,247.000000,147.700000,$0.000000
2,ocr_llm,gemini,gemini-3.1-flash-lite,41.7%,1.53,391.700000,166.000000,$0.003469
3,ocr_llm,lmstudio,google/gemma-4-e2b,40.0%,3.02,406.700000,102.800000,$0.000000


pipeline       llm_only             ocr_llm          
backend          gemini  lmstudio    gemini  lmstudio
file_name                                            
ticket_1.jpg   0.666667  0.333333  0.166667  0.166667
ticket_10.jpg  1.000000  1.000000  0.833333  0.833333
ticket_2.jpg   0.666667  0.500000  0.500000  0.500000
ticket_3.jpg   0.833333  0.333333  0.166667  0.166667
ticket_4.jpg   0.500000  0.333333  0.333333  0.333333
ticket_5.jpg   1.000000  0.500000  0.166667  0.166667
ticket_6.jpg   0.833333  0.666667  0.500000  0.500000
ticket_7.jpg   0.666667  0.666667  0.666667  0.666667
ticket_8.jpg   0.500000  0.500000  0.333333  0.333333
ticket_9.jpg   0.666667  0.333333  0.500000  0.333333

Saved: /Users/andrei/Documents/Projects/Curso LLM Codes/session_2_architecture/data/outputs/receipt_comparison_20260711T161623Z.csv


## 6. Dashboard final

La siguiente celda inicia Streamlit usando el mismo Python 3.12 del kernel. No requiere
JupyterLab ni una terminal externa. El comando equivalente es:

```bash
uv run streamlit run app.py
```

El dashboard permite inspeccionar el ground truth, comparar modelos y revisar el JSON por ticket.

<details><summary><strong>Decisión empresarial para revelar al final</strong></summary>

El modelo ganador no es necesariamente el más preciso. Documenten el umbral mínimo de calidad,
restricciones de privacidad, latencia operativa y costo mensual antes de registrar la decisión.
</details>

In [11]:
import socket
import subprocess
import sys
import time

DASHBOARD_PORT = 8501

def port_is_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as connection:
        return connection.connect_ex(("127.0.0.1", port)) == 0

if port_is_open(DASHBOARD_PORT):
    print(f"Dashboard already running: http://localhost:{DASHBOARD_PORT}")
else:
    dashboard_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            str(ROOT / "app.py"),
            "--server.headless=true",
            f"--server.port={DASHBOARD_PORT}",
        ],
        cwd=ROOT,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    time.sleep(2)
    print(f"Dashboard started: http://localhost:{DASHBOARD_PORT}")
    print(f"Process id: {dashboard_process.pid}")

Dashboard started: http://localhost:8501
Process id: 59784


---

## 📌 Nota final: OCR vs. LLM directo en este caso de uso

> **Hallazgo de las pruebas.** El pipeline `ocr_llm` (Tesseract + LLM) **sí disminuye costos**
> —el texto plano usa muchos menos tokens que la imagen— pero tiene **peor desempeño** de forma
> consistente en este dataset.
>
> Este es justo el caso donde **ni la estrategia híbrida recomendada conviene**: los tickets
> son fotos poco claras, poco nítidas y mal tomadas (arrugas, ángulo, iluminación), y el OCR
> tradicional degrada la información antes de que el LLM pueda interpretarla. El modelo
> multimodal, en cambio, "ve" el ticket completo y resuelve contexto que el OCR pierde.
>
> Después de las pruebas ya hay **indicios suficientes para justificar el LLM directo
> (`llm_only`)**: aunque el costo por ticket prácticamente **se duplica**, la mejora es muy
> significativa — **menos de la mitad de los errores** que el pipeline con OCR.
>
> **Lección para la decisión empresarial:** el costo por ticket no se evalúa solo; se evalúa
> contra el costo de un error (un RFC mal validado, un monto mal registrado). Cuando duplicar
> centavos por ticket reduce los errores a menos de la mitad, el LLM directo se justifica.
> La estrategia híbrida sigue siendo válida — pero para documentos escaneados con buena
> calidad, no para fotos de tickets como estas.
